# 🛰️ Landsat / Sentinel-2 Overpass Prediction Notebook: Overview
## **ROI Definition → Orbit Propagation → Swath Intersection → UTC/Local-Time Report**

This notebook is the **core prediction engine** of the workflow. Give it a Region of Interest and a time window and it tells you **when Landsat 8, Landsat 9 and Sentinel-2A/B/C will image that area**, how much of the ROI each pass covers, and when that happens in both **UTC** and the **ROI's own local time**, then writes the result to CSV.

> 🧭 **How this differs from n2yo.com**: n2yo reports when a satellite is *above your horizon*, which for a polar orbiter happens several times a day and says nothing about whether your area is imaged. This workflow intersects each satellite's **instrument swath** (Sentinel-2 MSI ≈ 290 km, Landsat OLI ≈ 185 km) with your ROI, so every row is a pass in which the sensor footprint actually covers the ROI.

---

### 🚀 Core Functional Areas
* **Orbit Propagation**: Current TLEs from CelesTrak, propagated with SGP4 (Skyfield) in a 30 s coarse scan followed by a 1 s refinement.
* **Swath-Aware Imaging Test**: Cross-track scan lines are built geodesically from the ground-track heading and intersected with the ROI in an ROI-centred equal-area projection, giving entry/exit times and true **% ROI coverage**.
* **Acquisition Realism**: Keeps daytime **descending** passes inside each mission's latitude limits; optionally confirms Sentinel-2 passes against **ESA's published acquisition plans**.
* **Time-Zone Intelligence**: Resolves the ROI's IANA time zone and converts every pass with the DST offset that applies on that date.
* **Reporting**: Summary tables, countdowns, Esri footprint maps, a timeline chart, CSV and GeoPackage exports.

## ⚙️ Workspace Configuration & Dependency Imports
> ### **Standard Library Initialisation → Custom Module Integration**

This configures the system path to recognise the local `Tools` directory and imports the orchestration functions from the **Overpass Prediction Workflow** utility.

---

### 📂 Orchestration Components
| Function | Responsibility |
| :--- | :--- |
| **`load_roi`** | Reads any vector file (any CRS), a shapely geometry, a `(lon, lat)` point or a bounding box, and returns one dissolved EPSG:4326 geometry. |
| **`visualise_roi_on_map`** | Interactive Folium map of the ROI for spatial verification. |
| **`resolve_roi_timezone`** | Resolves the IANA time zone at the ROI (e.g. `Australia/Melbourne`) so local times are DST-correct. |
| **`fetch_tles`** | Retrieves TLEs from CelesTrak (mirror fallback) with a 12 h on-disk cache and offline fallback. |
| **`predict_overpasses`** | SGP4 propagation + swath/ROI intersection → one row per imaging overpass. |
| **`summarise_overpasses` / `next_overpasses`** | Human-readable pass table and an n2yo-style countdown per satellite. |
| **`visualise_overpass_footprints` / `plot_overpass_timeline`** | Swath strips on Esri imagery, and a pass timeline chart. |
| **`fetch_s2_acquisition_plans` / `load_s2_acquisition_plans` / `crosscheck_s2_acquisition_plan`** | Confirms Sentinel-2 passes against ESA's published acquisition plans. |
| **`export_overpass_csv` / `export_overpass_footprints` / `export_live_tracker_html`** | CSV report (UTC + local), GeoPackage footprints, standalone live-tracker page. |

In [ ]:
# Core
import sys
from pathlib import Path

# Data
import pandas as pd

# Ensure the 'Tools' directory is in the path for custom module discovery
sys.path.insert(0, "Tools")

from overpass_helper_utility import (
    SATELLITE_CATALOGUE,
    load_roi,
    visualise_roi_on_map,
    resolve_roi_timezone,
    fetch_tles,
    predict_overpasses,
    summarise_overpasses,
    next_overpasses,
    visualise_overpass_footprints,
    plot_overpass_timeline,
    fetch_s2_acquisition_plans,
    load_s2_acquisition_plans,
    crosscheck_s2_acquisition_plan,
    export_overpass_csv,
    export_overpass_footprints,
    export_live_tracker_html,
    process_timer
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

## 🗺️ Region of Interest (ROI) Initialisation & Spatial Validation
> ### **Vector Data Ingestion → Interactive Cartographic Verification**

Define the ROI and check it sits where you expect. `load_roi` accepts any vector file GeoPandas can read (any CRS), a shapely geometry, a WKT string, a **point** `(lon, lat)` or a **bounding box** `(min_lon, min_lat, max_lon, max_lat)`.

---

### 📂 Logic & Orchestration Components
| Component | Responsibility | Analytical Objective |
| :--- | :--- | :--- |
| **`ROI_SHP_PATH`** | **Universal Path Resolution** | `Path` objects keep the notebook portable between Windows, macOS and Linux. |
| **`ROI_GDF`** | **Vector Data Ingestion** | Multi-part inputs are dissolved and repaired into one EPSG:4326 geometry. |
| **`buffer_km`** | **Edge Capture (optional)** | Expands the ROI, e.g. to list scenes that only clip its edge. |
| **`visualise_roi_on_map`** | **Spatial Sanity Check** | Overlays the ROI on Esri World Imagery before any prediction is run. |

> 💡 **Tip**: For a single site (a buoy, a field plot, a jetty) use a point: `load_roi((144.96, -37.82), name="Melbourne_CBD")`. Coverage is then reported as 0 or 100 %.

In [ ]:
# 1. Define your ROI (example ROI shipped with the repository)
ROI_SHP_PATH = Path("DATA") / "SHP" / "Port_Phillip_Bay_Extent_EPSG_32755.shp"
ROI_GDF = load_roi(ROI_SHP_PATH, name="Port_Phillip_Bay")

# Alternatives:
# ROI_GDF = load_roi((144.96, -37.82), name="Melbourne_CBD")                        # point
# ROI_GDF = load_roi((144.35, -38.35, 145.15, -37.80), name="Port_Phillip_Bay")     # bounding box

# 2. Visualise it (optional sanity check)
display(visualise_roi_on_map(ROI_GDF, tiles="Esri.WorldImagery"))
print(ROI_GDF.geometry.iloc[0].wkt[:200])

## 🕒 Local Time-Zone Resolution
> ### **ROI Location → IANA Zone → DST-Correct Conversion**

The ROI's time zone is looked up offline with `timezonefinder` and stored as an **IANA zone name**, not a fixed offset. A window that crosses a daylight-saving change (e.g. AEST ↔ AEDT on the first Sunday of April/October in Victoria) is converted correctly pass by pass. If the ROI straddles zones a warning is printed; force a zone with `override="Australia/Sydney"`. Offshore ROIs resolve to the nautical `Etc/GMT±N` zones.

In [ ]:
ROI_TZ = resolve_roi_timezone(ROI_GDF)          # or resolve_roi_timezone(ROI_GDF, override="UTC")

## 📡 Orbital Element Retrieval (TLE)
> ### **CelesTrak GP Service → Local Cache → Checksum-Verified Elements**

Two-line element sets (TLEs) are the orbital parameters SGP4 propagates. They are refreshed several times a week by the US Space Force and redistributed by CelesTrak.

---

### 📂 Retrieval Behaviour
| Behaviour | Detail |
| :--- | :--- |
| **Caching** | Each TLE is cached in `DATA/TLE/<norad>.tle` and reused for `max_age_hours` (default 12 h), as CelesTrak asks. |
| **Fallback** | If CelesTrak is unreachable the `tle.ivanstanojevic.me` mirror is tried, then the newest cache is used with a warning. |
| **Manual TLEs** | Paste TLE text (e.g. from n2yo or Space-Track) into `tle_text=`. Column spacing collapsed by web pages is rebuilt and checksum-verified. |
| **Age flag** | TLEs older than 7 days print in amber. Prediction error grows with TLE age — refresh before relying on passes weeks ahead. |

---

### 🛰️ Tracked Missions (`SATELLITE_CATALOGUE`)
| Satellite | NORAD | Sensor | Swath | Note |
| :--- | :--- | :--- | :--- | :--- |
| Landsat 8 | 39084 | OLI/TIRS | 185 km | 16-day repeat; 8 days offset from Landsat 9 |
| Landsat 9 | 49260 | OLI-2/TIRS-2 | 185 km | 16-day repeat |
| Sentinel-2A | 40697 | MSI | 290 km | **Extension campaign to 31 Dec 2026** — acquires selected regions only |
| Sentinel-2B | 42063 | MSI | 290 km | 10-day repeat; nominal pair with S2C (5-day combined) |
| Sentinel-2C | 60989 | MSI | 290 km | 10-day repeat |

In [ ]:
TLES = fetch_tles(
    satellites=None,                    # all of SATELLITE_CATALOGUE; or e.g. ["L8", "L9", "S2B", "S2C"]
    tle_dir=Path("DATA") / "TLE",
    max_age_hours=12,
    tle_text=None,                      # paste TLE text here to use your own elements
)

## 🚀 Prediction Parameter Configuration
> ### **Temporal Window → Acquisition Realism → Coverage Constraints**

The same `prediction_config` dictionary drives the prediction cell below.

---

### 📂 Configuration Parameters
| Parameter | Default/Value | Responsibility |
| :--- | :--- | :--- |
| **`start_utc`** | `None` (now) | Window start; strings without an offset are read as UTC. |
| **`days`** | `16` | Window length — one Landsat repeat cycle, 1.6 Sentinel-2 cycles. |
| **`daytime_descending_only`** | `True` | Keep only passes where optical imaging is expected (descending node, sun above `min_sun_elevation_deg`, inside the mission's latitude limits). Set `False` to also list night/ascending geometric passes, flagged ✘. |
| **`min_sun_elevation_deg`** | `0.0` | Solar elevation at the ROI below which a pass is not expected to be imaged. |
| **`min_roi_coverage_pct`** | `0.0` | Drop passes that cover less of the ROI (e.g. `25` to ignore edge slivers). |
| **`satellites`** | `None` | Subset, e.g. `["S2B", "S2C"]`. |
| **`fine_step_s`** | `1.0` | Time resolution of entry/exit; the closest approach is refined below it. |

In [ ]:
prediction_config = {
    "roi_gdf": ROI_GDF,
    "tles": TLES,
    "start_utc": None,                    # e.g. "2026-11-01 00:00"
    "days": 16,
    "satellites": None,
    "timezone_name": ROI_TZ,
    "daytime_descending_only": True,
    "min_sun_elevation_deg": 0.0,
    "min_roi_coverage_pct": 0.0,
    "fine_step_s": 1.0,
}

## 🔭 Overpass Prediction
> ### **Coarse Orbit Scan → Fine Refinement → Swath ∩ ROI → Attribution**

---

### 📋 Output Columns (selected)
| Column | Meaning |
| :--- | :--- |
| **`roi_entry_utc` / `roi_exit_utc`** | First and last instant the instrument swath touches the ROI. |
| **`overpass_utc`** | Closest approach of the ground track to the ROI anchor point — ≈ scene-centre time for a small ROI. |
| **`*_local`, `timezone`, `utc_offset`** | The same instants in the ROI's local time, with the offset in force on that date. |
| **`roi_coverage_pct`** | Share of the ROI area inside the swath strip on that pass. |
| **`nadir_offset_km`, `roi_side_of_track`, `view_angle_deg`** | How far off-nadir the ROI is imaged, on which side, and at what view angle. |
| **`edge_margin_km`** | Half swath minus nadir offset. Within ~±5 km of 0 the pass is marginal: the real swath edge differs slightly from the nominal width. |
| **`max_elevation_deg`** | Satellite elevation seen from the ROI (the number n2yo reports). |
| **`sun_elevation_deg`** | Solar elevation at the ROI at overpass time. |
| **`tle_age_days`** | Time between TLE epoch and the pass; larger means less certain timing. |

In [ ]:
with process_timer("Overpass Prediction"):
    PASSES = predict_overpasses(**prediction_config)

display(summarise_overpasses(PASSES))

### ⏱️ Next Overpass per Satellite
An n2yo-style countdown to each satellite's next imaging pass over the ROI.

In [ ]:
display(next_overpasses(PASSES))

## 🗺️ Footprint Map & Pass Timeline
> ### **Swath Strips on Esri Imagery → Calendar View**

Each strip is the instrument footprint for one pass, clipped to ~±150 km around the ROI, one toggleable layer per satellite. Hover for local/UTC time, direction, coverage and view angle. The timeline puts every pass on the local calendar; marker size scales with ROI coverage.

> 📌 **Viewing on GitHub?** Folium maps are HTML/JavaScript outputs that GitHub's notebook preview does not execute (*"Make this Notebook Trusted to load map"*). Run the notebook locally, or export with `save_path=`.

In [ ]:
display(visualise_overpass_footprints(PASSES, ROI_GDF, tiles="Esri.WorldImagery", expected_only=True))

In [ ]:
fig = plot_overpass_timeline(PASSES)

## 🗓️ Sentinel-2 Acquisition-Plan Cross-Check (Optional)
> ### **ESA Mission Planning KML → Planned Data-Takes → Pass Confirmation**

Orbital geometry says a satellite *can* image the ROI. Whether it *will* depends on the mission plan: Sentinel-2 does not acquire open ocean far from land, and **Sentinel-2A is in an extension campaign (to 31 Dec 2026) that covers selected regions only**. ESA publishes the planned data-takes for each unit as KML files roughly two weeks ahead.

---

### 📂 Plan Status Values
| Status | Meaning |
| :--- | :--- |
| **`planned (NOMINAL)`** | A nominal data-take of that unit overlaps the pass and intersects the ROI — data will be distributed. |
| **`planned (<other>)`** | Planned in a non-distributed mode (e.g. vicarious calibration). |
| **`not planned`** | The plan covers that date but no data-take over the ROI — no product expected. |
| **`no plan loaded`** | No downloaded plan covers that date yet (beyond ESA's planning horizon). |
| **`n/a`** | Landsat — USGS does not publish a comparable plan; Landsat acquires ~all daytime land and near-coastal WRS-2 scenes. |

> ⚠️ **Note**: If the ESA page layout changes and the download fails, save the KML files manually from the [acquisition-plans page](https://sentinels.copernicus.eu/copernicus/sentinel-2/acquisition-plans) into `DATA/KML/` and re-run — any KML found there is used.

In [ ]:
KML_DIR = Path("DATA") / "KML"
try:
    fetch_s2_acquisition_plans(KML_DIR,
                               start_utc=PASSES.attrs["window_start_utc"],
                               end_utc=PASSES.attrs["window_end_utc"])
except Exception as e:
    print(f"\x1b[93m⚠ Plan download failed ({type(e).__name__}: {e}). Using any KMLs already in {KML_DIR}.\x1b[0m")

PLANS = load_s2_acquisition_plans(KML_DIR)
if len(PLANS):
    PASSES = crosscheck_s2_acquisition_plan(PASSES, PLANS, ROI_GDF)
    display(summarise_overpasses(PASSES))
else:
    print("No acquisition plans available — continuing with geometric predictions only.")

## 💾 Report Export
> ### **CSV (UTC + Local) → GeoPackage Footprints → Live-Tracker Page**

---

### 📋 Export Behaviour
* **CSV**: UTC columns are ISO-8601 with `Z`; local columns carry their explicit offset (`2026-10-26T11:27:30+11:00`), plus split `local_date`, `local_time` and `local_weekday` columns for spreadsheets. File name: `OUTPUT/overpasses_<roi>_<start>_<end>.csv`.
* **GeoPackage**: Swath strips with all attributes, for QGIS / ArcGIS.
* **Live tracker**: A standalone HTML page that propagates the orbits in the browser and counts down to these passes (see `p02_overpass_live_tracker.ipynb`).

In [ ]:
OUTPUT_DIR = Path("OUTPUT")

csv_path = export_overpass_csv(PASSES, OUTPUT_DIR, expected_only=False)
gpkg_path = export_overpass_footprints(PASSES, OUTPUT_DIR / f"{csv_path.stem}_footprints.gpkg")
html_path = export_live_tracker_html(TLES, ROI_GDF, OUTPUT_DIR / "live_tracker.html", passes=PASSES)

display(pd.read_csv(csv_path).head())